In [1]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import os
import random
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from torchvision.utils import make_grid
from engression.models import StoNet, StoLayer
from engression.loss_func import energy_loss_two_sample
import argparse
import json
import xarray as xr
import torch.nn as nn
from sklearn.manifold import TSNE

import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import sys
sys.path.append('/home/sc.uni-leipzig.de/fl53wumy/llaae_new/DistributionalPrincipalAutoencoder')
import utils as ut

In [ ]:
def remove_nan_columns(tensor: torch.Tensor):
    """
    Removes columns that are entirely NaN from a 2D tensor.
    
    Returns:
        - reduced_tensor: Tensor with NaN-only columns removed
        - column_mask: Boolean mask indicating which columns were kept
    """
    if tensor.dim() != 2:
        raise ValueError("Input must be a 2D tensor")
        
    column_mask = ~torch.isnan(tensor).all(dim=0)
    reduced_tensor = tensor[:, column_mask]
    return reduced_tensor, column_mask
    
def restore_nan_columns(reduced_tensor: torch.Tensor, column_mask: torch.Tensor):
    """
    Restores removed NaN-only columns to a tensor using the original column mask.
    
    Returns:
        - restored_tensor: Tensor with original number of columns (NaNs in removed positions)
    """
    n_rows = reduced_tensor.shape[0]
    n_cols = column_mask.numel()
    
    restored_tensor = torch.full((n_rows, n_cols), float('nan'), dtype=reduced_tensor.dtype, device=reduced_tensor.device)
    restored_tensor[:, column_mask] = reduced_tensor
    return restored_tensor

def plot_temperature_panel(ax, dataarray, vmax_shared, sample_nr=None, title=""):
    """
    Plot a single temperature panel on a given axis with Cartopy.

    Parameters:
    ----------
    ax : matplotlib.axes._subplots.AxesSubplot
        Axis on which to plot.
    dataarray : xarray.DataArray
        The temperature data to plot.
    title : str
        Title for the subplot.
    vmax_shared : float
        Symmetric max value for colormap scaling (vmin = -vmax).
    """
    levels = np.linspace(-5, 5, 11)

    p = dataarray.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='coolwarm',
        vmin=-vmax_shared,
        vmax=vmax_shared,
        levels=levels,
        add_colorbar=False
    )
    ax.set_title(title)
    ax.coastlines(resolution='110m', linewidth=1)
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3)
    #ax.gridlines(draw_labels=False)
    if sample_nr is not None:
        ax.text(
            x=0, y=0.5,                # left edge, center vertically
            s=sample_nr,          # your annotation text
            transform=ax.transAxes,   # interpret x/y in axis coordinates (0–1)
            rotation=90,              # rotate to align with y-axis
            va='center', ha='right',  # vertical/horizontal alignment
            fontsize=12
        )
    return p

def torch_to_dataarray(x_tensor, coords_ds, lat_dim=32, lon_dim=32, name="variable"):
    """
    Convert a flattened 2D torch tensor to a 3D xarray.DataArray (lat, lon, time).

    Parameters:
    ----------
    x_tensor : torch.Tensor
        A 2D tensor of shape (time_steps, lat_dim * lon_dim), or a 1D tensor to be reshaped.
    lat_dim : int
        Number of latitude points.
    lon_dim : int
        Number of longitude points.
    coords_ds : xarray.Dataset
        Dataset containing 'lat', 'lon', and 'time' coordinates to assign.
    name : str, optional
        Name of the variable in the DataArray.

    Returns:
    -------
    xarray.DataArray
        The reshaped and labeled data as an xarray.DataArray.
    """
    # Step 1: Convert to NumPy
    data_np = x_tensor.detach().cpu().numpy()

    # Step 2: Determine time_steps and reshape
    time_steps = data_np.shape[0]
    data_np = data_np.reshape(time_steps, lat_dim, lon_dim)

    # Step 3: Transpose to (lat, lon, time)
    data_np = data_np.transpose(1, 2, 0)

    # Step 4: Create the DataArray
    da = xr.DataArray(
        data_np,
        dims=("lat", "lon", "time"),
        coords={
            "lat": coords_ds.lat,
            "lon": coords_ds.lon,
            "time": np.arange(time_steps)
        },
        name=name
    )

    return da

def standardize_numpy(X):
    mean = X.mean(axis=0, keepdims=True)
    std = X.std(axis=0, keepdims=True)
    return (X - mean) / (std), mean, std
    
def visual_sample(model_enc, model_dec, model_pred, x, y, save_dir, to_img=True):
    model_enc.eval()
    model_dec.eval()
    model_pred.eval()
    with torch.no_grad():
        # gen = model(x).detach().cpu()#.view(x.shape[0], 1, 128, 128)
        rec1 = model_dec(model_enc(y)).detach().cpu()
        rec2 = model_dec(model_enc(y)).detach().cpu()
            
        gen1 = model_dec(model_pred(x)).detach().cpu()
        gen2 = model_dec(model_pred(x)).detach().cpu()
    if to_img:
        rec1 = rec1.view(x.shape[0], 1, 128, 128)
        rec2 = rec2.view(x.shape[0], 1, 128, 128)
        gen1 = gen1.view(x.shape[0], 1, 128, 128)
        gen2 = gen2.view(x.shape[0], 1, 128, 128)
        y = y.view(y.shape[0], 1, 128, 128)
    # y = y.cpu().view(y.shape[0], 1, 128, 128)
    sample = torch.cat([y.cpu(), rec1, rec2, gen1, gen2])
    sample = torch.clamp(sample, torch.quantile(y, 0.005).item(), torch.quantile(y, 0.995).item())
    plt.matshow(make_grid(sample, nrow=y.shape[0]).permute(1, 2, 0)[:,:,0], cmap="Spectral_r"); plt.axis('off'); 
    plt.savefig(save_dir, bbox_inches="tight", pad_inches=0, dpi=300); plt.close()
    # save_image(sample, save_dir, normalize=True, scale_each=True)
    model_enc.train()
    model_dec.train()
    model_pred.train()

def data_to_torch(ds, variable):
    temp_data = ds[variable]
    data = temp_data.transpose('time', 'lat', 'lon')
    
    # Now convert to numpy
    data_np = data.values  # Shape: (time, lat, lon)
    
    # Flatten lat and lon together
    time_steps, lat_dim, lon_dim = data_np.shape
    data_np = data_np.reshape(time_steps, lat_dim * lon_dim)
    
    
    data_np = data_np  # Shape: (grid_cell, timestep)
    
    # Finally, convert to torch tensor
    data_tensor = torch.tensor(data_np, dtype=torch.float32)
    print(data_tensor.shape)
    return data_tensor

def predictors_to_torch(ds, variable):
    temp_data = ds#[variable]
    data = temp_data.transpose('time', 'mode')
    
    # Now convert to numpy
    data_np = data.values  # Shape: (time, lat, lon)

    
    # Flatten lat and lon together
    #time_steps, lat_dim, lon_dim = data_np.shape
    #data_np = data_np.reshape(time_steps, lat_dim * lon_dim)
    
    
    #data_np = data_np  # Shape: (grid_cell, timestep)
    
    # Finally, convert to torch tensor
    data_tensor = torch.tensor(data_np, dtype=torch.float32)
    print(data_tensor.shape)
    return data_tensor

In [2]:
random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

save_dir = "joint_low_level_out"

device: cpu


In [3]:
#### load data

# read contents from the settings.json file
settings_file_path = 'dpa_train_settings.json'

with open(settings_file_path, 'r') as file:
    settings = json.load(file)
  

# Save to a new file for logging
with open(f"used_settings.json", "w") as f:
    json.dump(settings, f, indent=4)

# load my temperature data
# Load your NetCDF file
ds = xr.open_dataset(settings['dataset_trefht'])
print("Dataset:", settings['dataset_trefht'])

ds_train = ds.isel(time=slice(0, 128000)) #4769 * 80
ds_test = ds.isel(time=slice(-64000, 476900)) #4769 * 80

# transform to torch tensors
x_tr = ut.data_to_torch(ds_train, "TREFHT")
x_te = ut.data_to_torch(ds_test, "TREFHT")

# load Z500
ds_z500_pre = xr.open_dataset(settings['dataset_z500'])
ds_z500, _, _ = ut.standardize_numpy(ds_z500_pre.pseudo_pcs.values)
print("z500 shape", ds_z500.shape)
z500 = torch.from_numpy(ds_z500) #predictors_to_torch(ds_z500, "pseudo_pcs")
print("z500 shape", z500.shape)
# scale Z500


z500_train = z500[:128000,:]#[:381520,:]
z500_test = z500[-64000:,:]

Dataset: /work/fl53wumy-llaae_data_new/fl53wumy-llaae_data_new-1748049607/llaae_data/europe_10percent_masked_stacked_TREFHT_JJA.nc
torch.Size([128000, 1024])
torch.Size([64000, 1024])
z500 shape (476900, 1001)
z500 shape torch.Size([476900, 1001])


In [4]:
# remove nans from data
x_tr_reduced, mask_x_tr = ut.remove_nan_columns(x_tr)
x_te_reduced, mask_x_te = ut.remove_nan_columns(x_te)
print(x_tr_reduced.shape)

torch.Size([128000, 648])


In [5]:
#### build model

# model parameters 
in_dim = x_tr_reduced.shape[1]
latent_dim = 20
num_layers = 6
hidden_dim = 100
hidden_dim_lm = 50
noise_dim = 20 
resblock = True
out_act=None
noise_dim_dec=20
beta = 1

# training parameters 
batch_size = 128 #190
bn = False
lr = 1e-4

# Encoder
model_enc = StoNet(in_dim=in_dim,
                   out_dim=latent_dim,
                   num_layer=num_layers,
                   hidden_dim=hidden_dim,
                   noise_dim=0,
                   add_bn=bn,
                   out_act=out_act,
                   resblock=resblock).to(device)
# Decoder
model_dec = StoNet(in_dim=latent_dim,
                   out_dim=in_dim,
                   num_layer=num_layers,
                   hidden_dim=hidden_dim,
                   noise_dim=noise_dim_dec,
                   add_bn=bn,
                   out_act=out_act,
                   resblock=resblock).to(device)
# Latent Map
model_pred = StoNet(in_dim=1000,
                    out_dim=latent_dim, 
                    num_layer=num_layers,
                    hidden_dim=hidden_dim_lm, 
                    noise_dim=noise_dim,
                    add_bn=bn, 
                    out_act=out_act, 
                    resblock=resblock).to(device)


optimizer = torch.optim.Adam(list(model_enc.parameters()) + list(model_dec.parameters()) + list(model_pred.parameters()), lr=lr)

In [6]:
print(x_te_reduced.shape)
print(x_tr_reduced.shape)
print(z500_train.shape)
print(z500_test.shape)

torch.Size([64000, 648])
torch.Size([128000, 648])
torch.Size([128000, 1001])
torch.Size([64000, 1001])


In [7]:
# prepare data

# create data loader Temperature
train_dataset = TensorDataset(z500_train, x_tr_reduced)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
print(f"Number of batches: {len(train_loader)}")

# create test loader Temperature
test_dataset = TensorDataset(z500_test, x_te_reduced)
test_loader_in = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
print(f"Number of batches: {len(test_loader_in)}")


Number of batches: 1000
Number of batches: 500


In [8]:
print(len((train_loader)))

1000


In [10]:
# training settings
print_every_nepoch = 1
plot_every_epoch = 5
training_epochs = 200


# log file
log_file_name = os.path.join(save_dir, 'log.txt')
log_file = open(log_file_name, "wt")
log = (
            f"Epoch, "
            f"Total loss, "
            f"Total S1, "
            f"Total S2, "
            f"DPA NRGY, "
            f"DPA s1, "
            f"DPA s2, "
            f"LM NRGY, "
            f"LM s1, "
            f"LM s1 ")
print(log)
log_file.write(log + '\n')
log_file.flush()

# Loss lists
epochs_list=[]

## total losses
loss_total_te=[]
loss_total_tr=[]

loss_s1_te_total=[]
loss_s2_te_total=[]
loss_s1_tr_total = []
loss_s2_tr_total = []

## latent map losses
loss_lin_lat_pred_te=[]
loss_lin_lat_pred_tr = []

loss_s1_lin_lat_pred_tr = []
loss_s2_lin_lat_pred_tr = []

loss_s1_lin_lat_pred_te = []
loss_s2_lin_lat_pred_te = []


## DPA losses
loss_dpa_te=[]
loss_dpa_tr=[]

loss_s1_dpa_tr = []
loss_s2_dpa_tr = []

loss_s1_dpa_te = []
loss_s2_dpa_te = []





#losses_lin_lat_pred_te = []
#losses_s1_lat_dec_te = []
#losses_s2_lat_dec_te = []


for epoch_idx in range(0, training_epochs):

    # total losses
    loss_tr = 0
    loss_te = 0
    s1_tr = 0
    s2_tr = 0
    s1_t2 = 0
    s2_t2 = 0

    # latent map
    loss_tr_pred = 0
    s1_tr_pred = 0
    s2_tr_pred = 0
    loss_te_pred = 0
    s1_te_pred = 0
    s2_te_pred = 0

    # DPA
    loss_tr_dpa = 0
    s1_tr_dpa = 0
    s2_tr_dpa = 0
    loss_te_dpa = 0
    s1_te_dpa = 0
    s2_te_dpa = 0
    
    
    for batch_idx, train_batch in enumerate(train_loader):
        optimizer.zero_grad()
        #x, y = data2pair(data_batch, device=device, rank=args.rank)
        
        # FL
        # get data
        #print("train batch length", len(train_batch))
        #print("y batch lenght", len(data_batch))

        x = train_batch[0].to(device)
        y = train_batch[1].to(device)
        #print("xshape", x.shape)
        print("yshape", y.shape)
        
        e = model_enc(y)
        rec1 = model_dec(e)
        print("rec1 shape:", rec1.shape)
        rec2 = model_dec(e)
        
        z1 = model_pred(x)
        gen1 = model_dec(z1)
        z2 = model_pred(x)
        gen2 = model_dec(z2)

        # FL
        # for plotting
        z3 = model_pred(x)
        gen3 = model_dec(z3)
        
        loss_rec, s1_rec, s2_rec = energy_loss_two_sample(y, rec1, rec2, verbose=True, beta=beta)
        
        #loss = loss_rec
        
        if False:
            loss_pred, s1_pred, s2_pred = energy_loss_two_sample(e, z1, z2, verbose=True, beta=beta) #loss im latent space
            loss_pre = loss_rec + loss_pred
            penalty_e = torch.linalg.vector_norm(e.std(dim=0) - 1) + torch.linalg.vector_norm(e.mean(dim=0)) # penalizes deviations from std=1 and mean = 0 for encoded sample 
            penalty_gen = torch.linalg.vector_norm(z1.std(dim=0) - 1) + torch.linalg.vector_norm(z1.mean(dim=0)) # penalizes deviations from std=1 and mean = 0 for predicted latent sample
            loss = loss_pre + penalty_e + penalty_gen
        else:
            loss_pred, s1_pred, s2_pred = energy_loss_two_sample(y, gen1, gen2, verbose=True, beta=beta)
            # welchen der beiden folgenden losses nutzen?
            # loss = loss_pred 
            loss = loss_rec + loss_pred
            
        loss.backward()
        optimizer.step()

        # loss total
        loss_tr += loss.item()
        s1_tr += s1_rec.item() + s1_pred.item()
        s2_tr += s2_rec.item() + s2_pred.item()

        # loss DPA
        loss_tr_dpa += loss_rec.item()
        s1_tr_dpa += s1_rec.item()
        s2_tr_dpa += s2_rec.item()

        # loss latent map
        loss_tr_pred += loss_pred.item()
        s1_tr_pred += s1_pred.item()            
        s2_tr_pred += s2_pred.item()
    
    if (epoch_idx == 0 or (epoch_idx + 1) % print_every_nepoch == 0):
        #log = f'[Epoch {epoch_idx + 1}]\tloss: {loss_tr / len(train_loader):.4f}, s1: {s1_tr / len(train_loader):.4f}, s2: {s2_tr / len(train_loader):.4f}'
        log = (
            f"Train {epoch_idx + 1}, "
            f"{loss_tr / len(train_loader):.4f}, "
            f"{s1_tr / len(train_loader):.4f}, "
            f"{s2_tr  / len(train_loader):.4f}, "
            f"{loss_tr_dpa / len(train_loader):.4f}, "
            f"{s1_tr_dpa / len(train_loader):.4f}, "
            f"{s2_tr_dpa / len(train_loader):.4f}, "
            f"{loss_tr_pred / len(train_loader):.4f}, "
            f"{s1_tr_pred / len(train_loader):.4f}, "
            f"{s2_tr_pred / len(train_loader):.4f}\n")

        
        #if epoch_idx == 0 or ((epoch_idx + 1) % (print_every_nepoch) == 0):
        model_enc.eval()
        model_dec.eval()
        model_pred.eval()
        loss_te = 0; loss_pred_te = 0; s1_te = 0; s2_te = 0
        with torch.no_grad():
            #for data_te in test_loader_in:
            #for batch_idx_test, (test_batch, slp_test_batch) in enumerate(zip(test_loader_in, test_loader_z500)):
            for batch_idx_test, test_batch in enumerate(test_loader_in):

                #x_te, y_te = data2pair(data_te, device=device, rank=args.rank)

                x_te = test_batch[0].to(device)
                y_te = test_batch[1].to(device)

                #print("xshape", x_te.shape)
                #print("yshape", y_te.shape)

                rec1_te = model_dec(model_enc(y_te))
                rec2_te = model_dec(model_enc(y_te))
                
                gen1_te = model_dec(model_pred(x_te))
                gen2_te = model_dec(model_pred(x_te))
                gen3_te = model_dec(model_pred(x_te))

                
                loss_te_rec_pre = energy_loss_two_sample(y_te, rec1_te, rec2_te, beta=beta)
                loss_te_pred_pre = energy_loss_two_sample(y_te, gen1_te, gen2_te, beta=beta)

                # total losses
                loss_te += loss_te_rec_pre[0].item()
                loss_te += loss_te_pred_pre[0].item()
                
                s1_te += loss_te_rec_pre[1].item() + loss_te_pred_pre[1].item() 
                s2_te += loss_te_rec_pre[2].item() + loss_te_pred_pre[2].item()

                # latent map
                loss_te_pred += loss_te_pred_pre[0].item()
                s1_te_pred += loss_te_pred_pre[1].item()
                s2_te_pred += loss_te_pred_pre[2].item()

                # DPA
                loss_te_dpa += loss_te_rec_pre[0].item()
                s1_te_dpa += loss_te_rec_pre[1].item()
                s2_te_dpa += loss_te_rec_pre[2].item()
                
        #log += f'\n  Test  \tloss: {loss_te / len(test_loader_in):.4f}, s1: {s1_te / len(test_loader_in):.4f}, s2: {s2_te / len(test_loader_in):.4f}'
        log += (
            f"Test {epoch_idx + 1}, "
            f"{loss_te / len(test_loader_in):.4f}, "
            f"{s1_te / len(test_loader_in):.4f}, "
            f"{s2_te  / len(test_loader_in):.4f}, "
            f"{loss_te_dpa / len(test_loader_in):.4f}, "
            f"{s1_te_dpa / len(test_loader_in):.4f}, "
            f"{s2_te_dpa / len(test_loader_in):.4f}, "
            f"{loss_te_pred / len(test_loader_in):.4f}, "
            f"{s1_te_pred / len(test_loader_in):.4f}, "
            f"{s2_te_pred / len(test_loader_in):.4f} ")
        model_enc.train()
        model_dec.train()
        model_pred.train()
            
        print(log)
        log_file.write(log + '\n')
        log_file.flush()
    
    ############
    ### PLOT ###
    ############
    if (epoch_idx == 0 or (epoch_idx + 1) % plot_every_epoch == 0):
        #print(batch_idx_test)
        if batch_idx_test == 499:
            #print("Now plotting samples ...")
            #print("Restore image shape ..")
            y_te_from_restored = ut.restore_nan_columns(y_te, mask_x_te)
            gen1_te_from_restored = ut.restore_nan_columns(gen1_te, mask_x_te)
            gen2_te_from_restored = ut.restore_nan_columns(gen2_te, mask_x_te)
            gen3_te_from_restored = ut.restore_nan_columns(gen3_te, mask_x_te)
    
    
            
            timesteps=[15, 42, 60, 94, 110]
            y_te_da = torch_to_dataarray(y_te_from_restored, coords_ds=ds_test, name="Temperature")
            reconstructed_da_11 = torch_to_dataarray(gen1_te_from_restored, coords_ds=ds_test, name="Temperature")
            reconstructed_da_12 = torch_to_dataarray(gen2_te_from_restored, coords_ds=ds_test, name="Temperature")
            reconstructed_da_21 = torch_to_dataarray(gen3_te_from_restored, coords_ds=ds_test, name="Temperature")
            
            fig, axs = plt.subplots(nrows=4, ncols=5, figsize=(20, 15), subplot_kw={'projection': ccrs.PlateCarree()})
            axs = axs.flatten()  # Flatten for easy indexing
            #print(axs)
            list_of_dataarrays_re = [reconstructed_da_11, reconstructed_da_12, reconstructed_da_21]
            for i, (ax, t) in enumerate(zip(axs, timesteps)):
                #title = f"{str(ds_test.isel(time=t).time.values)[:10]}"
                print("i:", i)
                if i ==0:
                    plot_temperature_panel(axs[0], y_te_da.isel(time=t), vmax_shared=5, sample_nr = "Truth")
                    plot_temperature_panel(axs[5], list_of_dataarrays_re[0].isel(time=t), vmax_shared=5, sample_nr = "Sample 1")
                    plot_temperature_panel(axs[10], list_of_dataarrays_re[1].isel(time=t), vmax_shared=5, sample_nr = "Sample 2")
                    plot_temperature_panel(axs[15], list_of_dataarrays_re[2].isel(time=t), vmax_shared=5, sample_nr = "Sample 3")
                    
                else:
                    plot_temperature_panel(ax, y_te_da.isel(time=t), vmax_shared=5)
                    plot_temperature_panel(axs[i+5], list_of_dataarrays_re[0].isel(time=t), vmax_shared=5)
                    plot_temperature_panel(axs[i+10], list_of_dataarrays_re[1].isel(time=t), vmax_shared=5)
                    plot_temperature_panel(axs[i+15], list_of_dataarrays_re[2].isel(time=t), vmax_shared=5)
                    
            
            
            
            
            # Optional: Add a colorbar
            cbar = fig.colorbar(axs[0].collections[0], ax=axs, orientation='horizontal', fraction=0.03, pad=0.05)
            cbar.set_label('Temperature (K)')
            
            
            #plt.savefig(f"{save_dir}epoch_{epoch_idx}_lat_dec_samples.pdf", format='pdf')
            plt.show()
        
    ############################
    ### Append to Loss Lists ###
    ############################
    epochs_list.append(epoch_idx)

    # total losses
    loss_total_tr.append(loss_tr / len(train_loader))
    loss_total_te.append(loss_te / len(test_loader_in))

    loss_s1_tr_total.append(s1_tr / len(train_loader))
    loss_s2_tr_total.append(s2_tr / len(train_loader))
        
    loss_s1_te_total.append(s1_te / len(test_loader_in))
    loss_s2_te_total.append(s2_te / len(test_loader_in))

    # latent map losses
    loss_lin_lat_pred_tr.append(loss_tr_pred / len(train_loader))
    loss_lin_lat_pred_te.append(loss_te_pred / len(test_loader_in))

    loss_s1_lin_lat_pred_tr.append(s1_tr_pred  / len(train_loader))
    loss_s2_lin_lat_pred_tr.append(s2_tr_pred / len(train_loader))
    
    loss_s1_lin_lat_pred_te.append(s1_te_pred/len(test_loader_in))
    loss_s2_lin_lat_pred_te.append(s2_te_pred/len(test_loader_in))

    ## DPA losses
    loss_dpa_te.append(loss_te_dpa / len(test_loader_in))
    loss_dpa_tr.append(loss_tr_dpa / len(train_loader))
    
    loss_s1_dpa_tr.append(s1_tr_dpa / len(train_loader))
    loss_s2_dpa_tr.append(s2_tr_dpa /len(train_loader))
    
    loss_s1_dpa_te.append(s1_te_dpa/len(test_loader_in))
    loss_s2_dpa_te.append(s2_te_dpa/len(test_loader_in))
    
    
    ############
    ### Plot ###
    ############
    # Dynamic plot
    # clear_output(wait=True)
    ut.plot_all_losses(
                    loss_total_tr, loss_total_te, loss_s1_tr_total, loss_s2_tr_total,
                    loss_s1_te_total, loss_s2_te_total,
                    loss_dpa_tr, loss_dpa_te, loss_s1_dpa_tr, loss_s2_dpa_tr,
                    loss_s1_dpa_te, loss_s2_dpa_te,
                    loss_lin_lat_pred_tr, loss_lin_lat_pred_te,
                    loss_s1_lin_lat_pred_tr, loss_s2_lin_lat_pred_tr,
                    loss_s1_lin_lat_pred_te, loss_s2_lin_lat_pred_te,
                    training_epochs)

Epoch, Total loss, Total S1, Total S2, DPA NRGY, DPA s1, DPA s2, LM NRGY, LM s1, LM s1 
yshape torch.Size([128, 648])
rec1 shape: torch.Size([128, 648])


RuntimeError: mat1 and mat2 shapes cannot be multiplied (128x1021 and 1020x50)

In [ ]:
# training settings
print_every_nepoch = 1
plot_every_epoch = 5
training_epochs = 200

# log file
log_file_name = os.path.join(save_dir, 'log.txt')
log_file = open(log_file_name, "wt")

# Loss lists
epochs_list=[]
loss_total_test=[]
losses_lin_lat_pred_te=[]
loss_s1_test=[]
loss_s2_test=[]

loss_total_train=[]
losses_lin_lat_pred_tr = []
losses_s1_tr = []
losses_s2_tr = []


#losses_lin_lat_pred_te = []
#losses_s1_lat_dec_te = []
#losses_s2_lat_dec_te = []


for epoch_idx in range(0, training_epochs):
    # FL
    loss_tr = 0
    loss_tr_pred = 0
    s1_tr = 0
    s1_tr_pred = 0
    s2_tr = 0
    s2_tr_pred = 0
    
    for batch_idx, train_batch in enumerate(train_loader):
        optimizer.zero_grad()
        #x, y = data2pair(data_batch, device=device, rank=args.rank)
        
        # FL
        # get data
        #print("train batch length", len(train_batch))
        #print("y batch lenght", len(data_batch))

        x = train_batch[0].to(device)
        y = train_batch[1].to(device)
        #print("xshape", x.shape)
        #print("yshape", y.shape)
        
        e = model_enc(y)
        rec1 = model_dec(e) 
        rec2 = model_dec(e)
        
        z1 = model_pred(x)
        gen1 = model_dec(z1)
        z2 = model_pred(x)
        gen2 = model_dec(z2)

        # FL
        # for plotting
        z3 = model_pred(x)
        gen3 = model_dec(z3)
        
        loss_rec, s1_rec, s2_rec = energy_loss_two_sample(y, rec1, rec2, verbose=True, beta=beta)
        
        loss = loss_rec
        
        if False:
            loss_pred, s1_pred, s2_pred = energy_loss_two_sample(e, z1, z2, verbose=True, beta=beta) #loss im latent space
            loss = loss + loss_pred
            penalty_e = torch.linalg.vector_norm(e.std(dim=0) - 1) + torch.linalg.vector_norm(e.mean(dim=0)) # penalizes deviations from std=1 and mean = 0 for encoded sample 
            penalty_gen = torch.linalg.vector_norm(z1.std(dim=0) - 1) + torch.linalg.vector_norm(z1.mean(dim=0)) # penalizes deviations from std=1 and mean = 0 for predicted latent sample
            loss = loss + penalty_e + penalty_gen
        else:
            loss_pred, s1_pred, s2_pred = energy_loss_two_sample(y, gen1, gen2, verbose=True, beta=beta)
            loss = loss_pred #loss + loss_pred
            
        loss.backward()
        optimizer.step()
        
        loss_tr += loss.item()
        s1_tr += s1_rec.item() + s1_pred.item()
        s2_tr += s2_rec.item() + s2_pred.item()
        
        loss_tr_pred += loss_pred.item()
        s1_tr_pred += s1_pred.item()            
        s2_tr_pred += s2_pred.item()
    
    if (epoch_idx == 0 or (epoch_idx + 1) % print_every_nepoch == 0):
        log = f'[Epoch {epoch_idx + 1}]\tloss: {loss_tr / len(train_loader):.4f}, s1: {s1_tr / len(train_loader):.4f}, s2: {s2_tr / len(train_loader):.4f}'
        
        if epoch_idx == 0 or ((epoch_idx + 1) % (print_every_nepoch) == 0):
            model_enc.eval()
            model_dec.eval()
            model_pred.eval()
            loss_te = 0; loss_pred_te = 0; s1_te = 0; s2_te = 0
            with torch.no_grad():
                #for data_te in test_loader_in:
                #for batch_idx_test, (test_batch, slp_test_batch) in enumerate(zip(test_loader_in, test_loader_z500)):
                for batch_idx_test, test_batch in enumerate(test_loader_in):

                    #x_te, y_te = data2pair(data_te, device=device, rank=args.rank)

                    x_te = test_batch[0].to(device)
                    y_te = test_batch[1].to(device)

                    #print("xshape", x_te.shape)
                    #print("yshape", y_te.shape)

                    rec1_te = model_dec(model_enc(y_te))
                    rec2_te = model_dec(model_enc(y_te))
                    
                    gen1_te = model_dec(model_pred(x_te))
                    gen2_te = model_dec(model_pred(x_te))
                    gen3_te = model_dec(model_pred(x_te))

                    
                    
                    loss_te += energy_loss_two_sample(y_te, rec1_te, rec2_te, beta=beta)[0].item()
                    loss_te += energy_loss_two_sample(y_te, gen1_te, gen2_te, beta=beta)[0].item()

                    loss_pred_te += energy_loss_two_sample(y_te, gen1_te, gen2_te, beta=beta)[0].item()
                    
                    s1_te += s1_rec.item() + s1_pred.item() 
                    s2_te += s2_rec.item() + s2_pred.item()
                    
            log += f'\n  Test  \tloss: {loss_te / len(test_loader_in):.4f}, s1: {s1_te / len(test_loader_in):.4f}, s2: {s2_te / len(test_loader_in):.4f}'
            model_enc.train()
            model_dec.train()
            model_pred.train()
            
        print(log)
        log_file.write(log + '\n')
        log_file.flush()
    
    if (epoch_idx == 0 or (epoch_idx + 1) % plot_every_epoch == 0):
        #print(batch_idx_test)
        if batch_idx_test == 499:
            #print("Now plotting samples ...")
            #print("Restore image shape ..")
            y_te_from_restored = restore_nan_columns(y_te, mask_x_te)
            gen1_te_from_restored = restore_nan_columns(gen1_te, mask_x_te)
            gen2_te_from_restored = restore_nan_columns(gen2_te, mask_x_te)
            gen3_te_from_restored = restore_nan_columns(gen3_te, mask_x_te)
    
    
            
            timesteps=[15, 42, 60, 94, 110]
            y_te_da = torch_to_dataarray(y_te_from_restored, coords_ds=ds_test, name="Temperature")
            reconstructed_da_11 = torch_to_dataarray(gen1_te_from_restored, coords_ds=ds_test, name="Temperature")
            reconstructed_da_12 = torch_to_dataarray(gen2_te_from_restored, coords_ds=ds_test, name="Temperature")
            reconstructed_da_21 = torch_to_dataarray(gen3_te_from_restored, coords_ds=ds_test, name="Temperature")
            ##########
            ###PLOT###
            ##########
            fig, axs = plt.subplots(nrows=4, ncols=5, figsize=(20, 15), subplot_kw={'projection': ccrs.PlateCarree()})
            axs = axs.flatten()  # Flatten for easy indexing
            #print(axs)
            list_of_dataarrays_re = [reconstructed_da_11, reconstructed_da_12, reconstructed_da_21]
            for i, (ax, t) in enumerate(zip(axs, timesteps)):
                #title = f"{str(ds_test.isel(time=t).time.values)[:10]}"
                print("i:", i)
                if i ==0:
                    plot_temperature_panel(axs[0], y_te_da.isel(time=t), vmax_shared=5, sample_nr = "Truth")
                    plot_temperature_panel(axs[5], list_of_dataarrays_re[0].isel(time=t), vmax_shared=5, sample_nr = "Sample 1")
                    plot_temperature_panel(axs[10], list_of_dataarrays_re[1].isel(time=t), vmax_shared=5, sample_nr = "Sample 2")
                    plot_temperature_panel(axs[15], list_of_dataarrays_re[2].isel(time=t), vmax_shared=5, sample_nr = "Sample 3")
                    
                else:
                    plot_temperature_panel(ax, y_te_da.isel(time=t), vmax_shared=5)
                    plot_temperature_panel(axs[i+5], list_of_dataarrays_re[0].isel(time=t), vmax_shared=5)
                    plot_temperature_panel(axs[i+10], list_of_dataarrays_re[1].isel(time=t), vmax_shared=5)
                    plot_temperature_panel(axs[i+15], list_of_dataarrays_re[2].isel(time=t), vmax_shared=5)
                    
            
            
            
            
            # Optional: Add a colorbar
            cbar = fig.colorbar(axs[0].collections[0], ax=axs, orientation='horizontal', fraction=0.03, pad=0.05)
            cbar.set_label('Temperature (K)')
            
            
            #plt.savefig(f"{save_dir}epoch_{epoch_idx}_lat_dec_samples.pdf", format='pdf')
            plt.show()
        
    epochs_list.append(epoch_idx)
    loss_total_train.append(loss_tr / len(train_loader))
    loss_total_test.append(loss_te / len(test_loader_in))
    
    losses_lin_lat_pred_tr.append(loss_tr_pred / len(train_loader))
    losses_lin_lat_pred_te.append(loss_pred_te / len(test_loader_in))
    
    losses_s1_tr.append(s1_tr / len(train_loader))
    losses_s2_tr.append(s2_tr / len(train_loader))
    
    loss_s1_test.append(s1_te / len(test_loader_in))
    loss_s2_test.append(s2_te / len(test_loader_in))
    
        
    
    # Dynamic plot
    # clear_output(wait=True)
    print(epochs_list)
    plt.figure(figsize=(8, 5))
    plt.plot(loss_total_train, label="Train: Total Loss", color='tab:blue')
    plt.plot(loss_total_test, label="Test: Total Loss", color='tab:orange')
    
    #plt.plot(losses_dpa_only_list_te, label="Test: DPA NRGY Loss", color='tab:orange')
    
    
    
    
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Live Training Loss")
    plt.legend()
    #plt.grid(True)
    #plt.ylim(0,30)
    #plt.xticks(ticks=range(0, training_epochs, 1))
    plt.xlim(0,training_epochs)
    plt.show()
    
    
    plt.figure(figsize=(8, 5))
    plt.plot(losses_lin_lat_pred_tr, label="Train: Pred. Loss", color='tab:red')
    plt.plot(losses_lin_lat_pred_te, label="Test: Pred. Loss", color='tab:green')
    plt.plot(losses_s1_tr, label = "Train S1", color='tab:blue')
    plt.plot(losses_s2_tr, label = "Train S2", color='tab:orange')
    plt.plot(loss_s1_test, label = "Test S1", color='tab:purple')
    plt.plot(loss_s2_test, label = "Test S2", color='tab:brown')
    
    
    
    
    
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Live Training Loss")
    plt.legend()
    #plt.grid(True)
    #plt.ylim(0,30)
    #plt.xticks(ticks=range(0, training_epochs, 1))
    plt.xlim(0,training_epochs)
    plt.show()
    
    
        


In [ ]:
# combined losses (DPA + latent map)
plt.figure(figsize=(8, 5))
plt.plot(loss_total_tr, label="Train: Total Loss", color='tab:blue')
plt.plot(loss_total_te, label="Test: Total Loss", color='tab:orange')
plt.plot(loss_s1_tr_total, label="Train: S1 Total Loss", color='tab:orange')
plt.plot(loss_s2_tr_total, label="Train: S2 Total Loss", color='tab:orange')
plt.plot(loss_s1_te_total, label="Test: S1 Total Loss", color='tab:orange')
plt.plot(loss_s1_te_total, label="Test: S2 Total Loss", color='tab:orange')


plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Combined ")
plt.legend()
plt.grid(True)
plt.xlim(0,training_epochs)
plt.show()



# DPA losses
plt.figure(figsize=(8, 5))
plt.plot(loss_dpa_tr, label="Train", color='tab:red')
plt.plot(loss_dpa_te, label="Test", color='tab:green')
plt.plot(loss_s1_dpa_tr, label = "Train S1", color='tab:blue')
plt.plot(loss_s2_dpa_tr, label = "Train S2", color='tab:orange')
plt.plot(loss_s1_dpa_te, label = "Test S1", color='tab:brown')
plt.plot(loss_s2_dpa_te, label = "Test S2", color='tab:purple')

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Latent Map Losses")
plt.legend()
plt.grid(True)
plt.xlim(0,training_epochs)
plt.show()


# latent map losses
plt.figure(figsize=(8, 5))
plt.plot(loss_lin_lat_pred_tr, label="Train", color='tab:red')
plt.plot(loss_lin_lat_pred_te, label="Test", color='tab:green')
plt.plot(loss_s1_lin_lat_pred_tr, label = "Train S1", color='tab:blue')
plt.plot(loss_s2_lin_lat_pred_tr, label = "Train S2", color='tab:orange')
plt.plot(loss_s1_lin_lat_pred_te, label = "Test S1", color='tab:brown')
plt.plot(loss_s2_lin_lat_pred_te, label = "Test S2", color='tab:purple')

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Latent Map Losses")
plt.legend()
plt.grid(True)
plt.xlim(0,training_epochs)
plt.show()